## Project setup
You can replace it with your GitHub repository to make making changes easier.

In [1]:
!rm -rf DiffusionPen
!git clone https://github.com/koninik/DiffusionPen
!cp -r DiffusionPen/* .

Cloning into 'DiffusionPen'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 199 (delta 107), reused 97 (delta 97), pack-reused 83 (from 1)
Receiving objects: 100% (199/199), 12.74 MiB | 1.98 MiB/s, done.
Resolving deltas: 100% (110/110), done.


### Data setup

In [7]:
!rm -r iam_data
!mkdir -p iam_data
!mkdir -p iam_data/words
!mkdir -p iam_data/ascii
!gdown --id 16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY
!gdown --id 187Ww-EIPU6SNwMlKWB3_WDlr8lix3xJs
!tar -xzf words.tgz -C iam_data/words
!tar -xzf ascii.tgz -C iam_data/ascii

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY
From (redirected): https://drive.google.com/uc?id=16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY&confirm=t&uuid=3267a3da-2018-4b32-a6d0-9cdd5ffb214f
To: /Users/ihorivanyshyn/Documents/ComputerVision/ivanyshyn_hw2/words.tgz
100%|████████████████████████████████████████| 821M/821M [02:05<00:00, 6.54MB/s]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=187Ww-EIPU6SNwMlKWB3_WDlr8lix3xJs
To: 

In [8]:
!ls ./iam_data/ascii/forms.txt

./iam_data/ascii/forms.txt


### Code update

In [10]:
!sed -i '' -e 's|/path/to/iam_data/|./iam_data/|g' style_encoder_train.py

## Fix Training Pipeline (5 points)

1. Fix the training pipeline.

2. Split the dataset into training and validation sets in advance using a fixed split.

3. Provide a summary describing:
  - what changes were made to the pipeline to make it work, and
  - how well the pipeline performs on the training and validation datasets.

4. Use the dataset https://www.kaggle.com/datasets/annyhnatiuk/ukrainian-handwritten-text
 to train a model for Ukrainian handwritten text recognition.

5. Submit link to Google Collab.

### Summary

Changes made to the pipeline:
- Replaced `random_split` with a fixed train/validation split using index lists stored in `iam_data/fixed_train_val_split.json` and loaded with `Subset` for reproducibility.
- I swapped out the long, hardcoded file path for a simple ./iam_data/ folder in the local directory.
- the critical fix was on 380 line, I have deleted "if img.width < 256" and used comented code, but with border value 255, the main logic of this fix is to make enable put images of different size in same batch.
- On line 415 change "img" to "s_img", because "s_img" it is styled image, and if we paste "img" we will paste same image over all cicle.
- Also in lines 429-439 add condition to take characters from the set, also standartize padding, because, 79 is something stange better to take max lenght, and also add "max_sequence_lenght" to make all char sequences same lenght.

After I have also add changes, to run it on Ukrainian dataset:
- it maps data, parse metadata, splita and cleans data.
- also add ukrainian characters to character set.
- I must fix  writer-id granularity for Ukrainian samples, to make no callapsing, when it goes to one writer.

Also all splits are saved in ".json" files

All results is in markdowns below, pipeline well performs on Train and Val 78-76%, and as we can see there is no overfitting, but perfomance is pure on test, I think thats because test performs on unseen data. Also test use disjoint writers and separate writer-id mapping, so its perfomance is not directly comporable to train and validation. On Ukrainian language it performs whorse - 48/28/30, a little overfitting to train data, but also val and test is almost the same.

In [1]:
!python3 style_encoder_train.py

num_tokens 1
Using device: mps
MPS detected: reducing batch size from 320 to 16 to avoid OOM.
save_file ./IAM_dataset_PIL_style/train_word_IAM.pt
root_path ./iam_data//words
imgs: [0/55558 (0%)]
imgs: [1000/55558 (2%)]
imgs: [2000/55558 (4%)]
imgs: [3000/55558 (5%)]
imgs: [4000/55558 (7%)]
imgs: [5000/55558 (9%)]
imgs: [6000/55558 (11%)]
imgs: [7000/55558 (13%)]
imgs: [8000/55558 (14%)]
imgs: [9000/55558 (16%)]
imgs: [10000/55558 (18%)]
imgs: [11000/55558 (20%)]
imgs: [12000/55558 (22%)]
imgs: [13000/55558 (23%)]
imgs: [14000/55558 (25%)]
imgs: [15000/55558 (27%)]
imgs: [16000/55558 (29%)]
imgs: [17000/55558 (31%)]
imgs: [18000/55558 (32%)]
imgs: [19000/55558 (34%)]
imgs: [20000/55558 (36%)]
imgs: [21000/55558 (38%)]
imgs: [22000/55558 (40%)]
imgs: [23000/55558 (41%)]
imgs: [24000/55558 (43%)]
imgs: [25000/55558 (45%)]
imgs: [26000/55558 (47%)]
imgs: [27000/55558 (49%)]
imgs: [28000/55558 (50%)]
imgs: [29000/55558 (52%)]
imgs: [30000/55558 (54%)]
imgs: [31000/55558 (56%)]
imgs: [32000/

In [6]:
!python3 style_encoder_train.py --mode mixed --dataset iam --model mobilenetv2_100 --device mps --batch_size 16 --num_workers 0 --test_only --checkpoint style_models/mixed_iam_mobilenetv2_100.pth

num_tokens 1
Using device: mps
save_file ./IAM_dataset_PIL_style/train_word_IAM.pt
root_path ./iam_data//words
imgs: [0/55558 (0%)]
imgs: [1000/55558 (2%)]
imgs: [2000/55558 (4%)]
imgs: [3000/55558 (5%)]
imgs: [4000/55558 (7%)]
imgs: [5000/55558 (9%)]
imgs: [6000/55558 (11%)]
imgs: [7000/55558 (13%)]
imgs: [8000/55558 (14%)]
imgs: [9000/55558 (16%)]
imgs: [10000/55558 (18%)]
imgs: [11000/55558 (20%)]
imgs: [12000/55558 (22%)]
imgs: [13000/55558 (23%)]
imgs: [14000/55558 (25%)]
imgs: [15000/55558 (27%)]
imgs: [16000/55558 (29%)]
imgs: [17000/55558 (31%)]
imgs: [18000/55558 (32%)]
imgs: [19000/55558 (34%)]
imgs: [20000/55558 (36%)]
imgs: [21000/55558 (38%)]
imgs: [22000/55558 (40%)]
imgs: [23000/55558 (41%)]
imgs: [24000/55558 (43%)]
imgs: [25000/55558 (45%)]
imgs: [26000/55558 (47%)]
imgs: [27000/55558 (49%)]
imgs: [28000/55558 (50%)]
imgs: [29000/55558 (52%)]
imgs: [30000/55558 (54%)]
imgs: [31000/55558 (56%)]
imgs: [32000/55558 (58%)]
imgs: [33000/55558 (59%)]
imgs: [34000/55558 (61%)

Training Accuracy: 78.4498

Validation Accuracy: 77.2118

Test Accuracy: 0.3545

In [2]:
!pip -q install transformers datasets accelerate evaluate jiwer kaggle


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

import torch

device = "mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

!python3 style_encoder_train.py --mode mixed --dataset ukr --eval_test_after_train


Using device: mps
num_tokens 1
Using device: mps
MPS detected: reducing batch size from 320 to 16 to avoid OOM.
save_file ./IAM_dataset_PIL_style/train_line_UKR.pt
Number of writers 396
100%|█████████████████████████████████| 25979/25979 [00:00<00:00, 818858.20it/s]
Character classes: ['\t', '\n', ' ', '!', '"', '%', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'B', 'I', 'O', 'Q', 'S', 'V', 'X', 'a', 'c', 'e', 'g', 'i', 'l', 'n', 'p', 'r', 'u', 'Є', 'І', 'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ж', 'З', 'Й', 'К', 'Л', 'М', 'Н', 'О', 'П', 'Р', 'С', 'Т', 'У', 'Ф', 'Х', 'Ц', 'Ч', 'Ш', 'Щ', 'Ю', 'Я', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ь', 'ю', 'я', 'є', 'і', 'ї', 'ґ', ' '] (105 different characters)
Max transcription length: 508
save_file ./IAM_dataset_PIL_style/val_line_UKR.pt
Number of writers 396
100%|██████████████████████████████████| 5566/5566 [00:00

In [3]:

!python3 style_encoder_train.py --mode mixed --dataset ukr --model mobilenetv2_100 --device {device} --batch_size 8 --num_workers 0 --max_test_samples 800 --test_only --checkpoint ./style_models/mixed_ukr_mobilenetv2_100.pth

num_tokens 1
Using device: mps
save_file ./IAM_dataset_PIL_style/train_line_UKR.pt
Number of writers 396
100%|█████████████████████████████████| 25979/25979 [00:00<00:00, 992357.44it/s]
Character classes: ['\t', '\n', ' ', '!', '"', '%', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'B', 'I', 'O', 'Q', 'S', 'V', 'X', 'a', 'c', 'e', 'g', 'i', 'l', 'n', 'p', 'r', 'u', 'Є', 'І', 'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ж', 'З', 'Й', 'К', 'Л', 'М', 'Н', 'О', 'П', 'Р', 'С', 'Т', 'У', 'Ф', 'Х', 'Ц', 'Ч', 'Ш', 'Щ', 'Ю', 'Я', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ь', 'ю', 'я', 'є', 'і', 'ї', 'ґ', ' '] (105 different characters)
Max transcription length: 508
save_file ./IAM_dataset_PIL_style/val_line_UKR.pt
Number of writers 396
100%|██████████████████████████████████| 5566/5566 [00:00<00:00, 1215341.56it/s]
Character classes: ['\t', '\n', ' ', '!', '"', '%', "'", 

Training Accuracy: 48.6239

Validation Accuracy: 28.3866

Test Accuracy: 30.6250